# Download and Preview Market Data

This notebook runs the first stage of the project: download raw equity history and option-chain data from Yahoo Finance, save the raw CSV files, and inspect the returned tables.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "src" / "bs_pricer").exists() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from bs_pricer.config import MarketConfig
from bs_pricer.pipeline import run_pipeline

PROJECT_ROOT
from bs_pricer.plotting import plot_volatility_smile, plot_volatility_surface

In [ ]:
config = MarketConfig(
    ticker="SPY",
    start_date="2023-01-01",
    end_date=None,
    raw_data_dir=PROJECT_ROOT / "data" / "raw",
    processed_data_dir=PROJECT_ROOT / "data" / "processed",
)

config

## Volatility Comparison Setup

For SPY, this notebook downloads equity history from 2023-01-01 onward. Later, realized volatility can be compared with option maturity buckets using common trading-day windows: 21 days for roughly 1 month, 63 days for roughly 3 months, and 126 days for roughly 6 months.

In [ ]:
result = run_pipeline(config)

equity_history = result["equity_history"]
option_chains = result["option_chains"]
equity_clean = result["equity_clean"]
options_clean = result["options_clean"]
options_with_iv = result["options_with_iv"]

(
    result["equity_path"],
    result["options_path"],
    result["equity_processed_path"],
    result["options_processed_path"],
    result["options_with_iv_path"],
)

In [ ]:
equity_history.shape, option_chains.shape

In [ ]:
options_clean.head()

In [ ]:
options_with_iv.head()

In [ ]:
price_col = "Adj Close" if "Adj Close" in equity_history.columns else "Close"

ax = equity_history[price_col].plot(
    figsize=(12, 5),
    title=f"{config.ticker} {price_col} Price",
)
ax.set_xlabel("Date")
ax.set_ylabel("Price")

In [ ]:
equity_history.tail()

In [ ]:
option_chains.head()

In [ ]:
option_chains.columns.tolist()

In [ ]:
option_chains.groupby(["expiry", "option_type"]).size().head(20)

In [ ]:
option_chains[["contractSymbol", "expiry", "option_type", "strike", "lastPrice", "bid", "ask", "volume", "openInterest", "impliedVolatility"]].head(20)

## Cleaned Data Preview

In [ ]:
equity_clean.head()

In [ ]:
options_clean.head()

In [ ]:
options_clean[["option_type", "expiry", "strike", "mid_price", "spot", "moneyness", "days_to_expiry", "time_to_expiry"]].head(20)

## Implied Volatility Preview

In [ ]:
options_with_iv.head()

In [ ]:
options_with_iv["solved_iv"].describe()

In [ ]:
options_with_iv[["option_type", "expiry", "strike", "mid_price", "spot", "moneyness", "time_to_expiry", "solved_iv"]].head(4000)

In [ ]:
options_with_iv[
    ["option_type", "expiry", "strike", "mid_price", "spot", "moneyness", "time_to_expiry", "solved_iv"]
].dropna(subset=["solved_iv"]).sort_values(["expiry", "option_type", "strike"]).head(50)

In [ ]:
options_with_iv[
    ["option_type", "expiry", "strike", "mid_price", "spot", "moneyness", "time_to_expiry", "solved_iv"]
].dropna(subset=["solved_iv"]).sample(20, random_state=1)

## Volatility Smile and Surface

In [ ]:
plot_volatility_smile(options_with_iv, option_type="call", x="moneyness")

In [ ]:
plot_volatility_surface(options_with_iv, option_type="call", x="moneyness")